In [1]:
%pip install -r requirements.txt

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
%pip install --force-reinstall "snowflake-connector-python[pandas]"

  Using cached snowflake_connector_python-5.0.0-cp312-cp312-win_amd64.whl.metadata (7.1 kB)
  Using cached cryptography-50.0.2-cp311-abi3-win_amd64.whl.metadata (4.4 kB)
  Using cached protobuf-6.33.6-cp310-abi3-win_amd64.whl.metadata (593 bytes)
  Using cached pytz-2026.5-py2.py3-none-any.whl.metadata (22 kB)
  Using cached tomlkit-0.15.1-py3-none-any.whl.metadata (3.3 kB)
  Using cached pandas-2.3.3-cp312-cp312-win_amd64.whl.metadata (19 kB)
  Using cached pyarrow-26.0.0-cp312-cp312-win_amd64.whl.metadata (3.0 kB)
  Using cached cffi-2.1.1-cp312-cp312-win_amd64.whl.metadata (2.6 kB)
  Using cached numpy-2.5.4-cp312-cp312-win_amd64.whl.metadata (6.6 kB)
  Using cached python_dateutil-2.9.0.post0-py2.py3-none-any.whl.metadata (8.4 kB)
  Using cached tzdata-2026.5-py2.py3-none-any.whl.metadata (1.4 kB)
  Using cached pycparser-3.11-py3-none-any.whl.metadata (8.2 kB)
  Using cached six-1.17.0-py2.py3-none-any.whl.metadata (1.7 kB)
Using cached cryptography-50.0.2-cp311-abi3-win_amd64.whl

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
from getpass import getpass
from agents.db import get_connection
from agents.planner import planner_agent
from agents.checker import checker_agent, get_live_alerts
from agents.retriever import search_policies

conn = get_connection()
cur = conn.cursor()
MBTA_KEY = getpass("MBTA key: ")
print("Ready ✅")

In [3]:
from agents.db import get_connection
from agents.planner import planner_agent

conn = get_connection()

c:\Users\agast\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\snowflake\connector\connection\_connection.py:135: FutureWarning: enable_server_session_keep_alive_auto_detection was not set and defaults to True. In a future version, the default will change to None. Please provide an explicit value: True = check for running async queries before logout (queries are preserved); False/None = always send logout on close (async queries may be terminated by server). Logout behavior can also be overridden with server_session_keep_alive. See the connection parameter docs for more info.
  options = self.config.to_proto_options(


In [4]:
print(planner_agent(conn, "Harvard", "Back Bay"))
print(planner_agent(conn, "Harvard", "Back Bay", avoid=["place-dwnxg"]))

{'ok': True, 'path': ['place-harsq', 'place-cntsq', 'place-knncl', 'place-chmnl', 'place-pktrm', 'place-dwnxg', 'place-chncl', 'place-tumnl', 'place-bbsta'], 'stations': ['Harvard', 'Central', 'Kendall/MIT', 'Charles/MGH', 'Park Street', 'Downtown Crossing', 'Chinatown', 'Tufts Medical Center', 'Back Bay'], 'transfers': ['Downtown Crossing'], 'transfer_ids': ['place-dwnxg']}
{'ok': False, 'reason': 'No route found avoiding those stations'}


In [5]:
print(planner_agent(conn, "Harvard", "Haymarket")["stations"])
print(planner_agent(conn, "Harvard", "Haymarket", avoid=["place-gover"])["stations"])

['Harvard', 'Central', 'Kendall/MIT', 'Charles/MGH', 'Park Street', 'Government Center', 'Haymarket']
['Harvard', 'Central', 'Kendall/MIT', 'Charles/MGH', 'Park Street', 'Downtown Crossing', 'State', 'Haymarket']


In [6]:
tests = [
    # (origin, destination, avoid, what you should see)
    ("Harvard", "Back Bay", None, "Red to Downtown Crossing, then Orange"),
    ("Wonderland", "Forest Hills", None, "Blue to State, then Orange"),
    ("Alewife", "Ashmont", None, "Red Line only, no transfers"),
    ("Ashmont", "Braintree", None, "must go through JFK/UMass (branch fix)"),
    ("Harvard", "Haymarket", ["place-gover"], "avoids Government Center"),
    ("harvard", "KENDALL/MIT", None, "works with any capitalization"),
    ("Hogwarts", "Harvard", None, "should fail: station not found"),
    ("Boylston", "Harvard", None, "should fail: not wheelchair accessible"),
]

for origin, dest, avoid, expected in tests:
    result = planner_agent(conn, origin, dest, avoid=avoid)
    print(f"{origin} -> {dest}" + (f"  (avoid {avoid})" if avoid else ""))
    print(f"  Expected: {expected}")
    if result["ok"]:
        print(f"  Route:     {' -> '.join(result['stations'])}")
        print(f"  Transfers: {result['transfers'] or 'none'}")
    else:
        print(f"  Failed:    {result['reason']}")
    print()

Harvard -> Back Bay
  Expected: Red to Downtown Crossing, then Orange
  Route:     Harvard -> Central -> Kendall/MIT -> Charles/MGH -> Park Street -> Downtown Crossing -> Chinatown -> Tufts Medical Center -> Back Bay
  Transfers: ['Downtown Crossing']

Wonderland -> Forest Hills
  Expected: Blue to State, then Orange
  Route:     Wonderland -> Revere Beach -> Beachmont -> Suffolk Downs -> Orient Heights -> Wood Island -> Airport -> Maverick -> Aquarium -> State -> Downtown Crossing -> Chinatown -> Tufts Medical Center -> Back Bay -> Massachusetts Avenue -> Ruggles -> Roxbury Crossing -> Jackson Square -> Stony Brook -> Green Street -> Forest Hills
  Transfers: ['State']

Alewife -> Ashmont
  Expected: Red Line only, no transfers
  Route:     Alewife -> Davis -> Porter -> Harvard -> Central -> Kendall/MIT -> Charles/MGH -> Park Street -> Downtown Crossing -> South Station -> Broadway -> Andrew -> JFK/UMass -> Savin Hill -> Fields Corner -> Shawmut -> Ashmont
  Transfers: none

Ashmont -

In [7]:
from getpass import getpass
from agents.checker import checker_agent, get_live_alerts

MBTA_KEY = getpass("MBTA key: ")

In [8]:
for a in get_live_alerts(MBTA_KEY)[:10]:
    print("-", a["attributes"]["effect"], "|", a["attributes"]["header"][:100])

- ESCALATOR_CLOSURE | Courthouse Escalator 417 (Airport/Design Center platform to south lobby) is unavailable due to ongoi
- ELEVATOR_CLOSURE | Natick Center Elevator 751 (Track 2 to street) is unavailable.
- ESCALATOR_CLOSURE | State Escalator 6 (Oak Grove platform to Old State House lobby) unavailable
- ELEVATOR_CLOSURE | Science Park/West End Elevator 981 (Lechmere & North platform to lobby) is unavailable due to mainte
- ACCESS_ISSUE | The Central Avenue mini-high platform (Ashmont side) is out of service. 
- ELEVATOR_CLOSURE | Back Bay Elevator 854 (Commuter Rail track 2 to lobby) is unavailable.
- ESCALATOR_CLOSURE | Alewife Escalator 351 (Main concourse to platform) is unavailable due to maintenance.
- ELEVATOR_CLOSURE | Wonderland Elevator 711 (Ocean Avenue to pedestrian plaza) is unavailable due to maintenance.
- ELEVATOR_CLOSURE | State Elevator 803 (Forest Hills platform to Washington Street, Milk Street) is unavailable due to m
- ELEVATOR_CLOSURE | Worcester Elevator 782 (P

In [9]:
plan = planner_agent(conn, "Harvard", "Back Bay")
check = checker_agent(plan, MBTA_KEY)

print("Route:", " → ".join(plan["stations"]))
print("Safe to travel:", check["safe"])
for p in check["all_issues"]:
    print(f"  [{p['role']}] {p['station']}: {p['effect']} — {p['alert'][:80]}")

Route: Harvard → Central → Kendall/MIT → Charles/MGH → Park Street → Downtown Crossing → Chinatown → Tufts Medical Center → Back Bay
Safe to travel: False
  [destination] Back Bay: ELEVATOR_CLOSURE — Back Bay Elevator 854 (Commuter Rail track 2 to lobby) is unavailable.
  [transfer] Downtown Crossing: ELEVATOR_CLOSURE — Downtown Crossing Elevator 869 (Alewife platform to Summer Street Concourse) is 


In [10]:
for p in check["blocking"]:
    print(p["station"], "→", p["alert"])

Back Bay → Back Bay Elevator 854 (Commuter Rail track 2 to lobby) is unavailable.
Downtown Crossing → Downtown Crossing Elevator 869 (Alewife platform to Summer Street Concourse) is unavailable due to maintenance through Tuesday, December 1.


In [11]:
print(planner_agent(conn, "Ashmont", "Braintree")["stations"])

['Ashmont', 'Shawmut', 'Fields Corner', 'Savin Hill', 'JFK/UMass', 'North Quincy', 'Wollaston', 'Quincy Center', 'Quincy Adams', 'Braintree']


In [1]:
from agents.replanner import replanner_agent

result = replanner_agent(conn, "Harvard", "Back Bay", MBTA_KEY)

if not result["ok"]:
    print("Failed:", result["reason"])
elif result["changed"]:
    print("Original:", " → ".join(result["original_plan"]["stations"]))
    print("Problems:", [p["station"] for p in result["problems"]])
    print("New route:", " → ".join(result["new_plan"]["stations"]) if result["new_plan"] else "none found")
    print("\nExplanation:\n", result["explanation"])
    print("\nSources:", result["sources"])
else:
    print(result["explanation"])

c:\Users\agast\AppData\Local\Python\pythoncore-3.12-64\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


NameError: name 'conn' is not defined